# 🔬 Shrink or Sink — Architecture Search Notebook
> **Runtime → Change runtime type → T4 GPU** before running!

This notebook runs the full search pipeline:
1. Install dependencies
2. Clone the repo
3. Show the ranked architecture search space
4. Train the Teacher (ResNet-18)
5. Run binary search to find smallest model ≥ 85%
6. Inspect and save results

## ⚙️ Step 0 — GPU Check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 📦 Step 1 — Clone Repo & Install Deps

In [ ]:
# ── Replace with your actual GitHub repo URL ───────────────────────────────
REPO_URL = 'https://github.com/ayushaanand/shrink_or_sink_25.git'
# ──────────────────────────────────────────────────────────────────────────

!git clone {REPO_URL} sos && mv sos/* . && rmdir sos
!pip install torch torchvision tqdm -q

## 📊 Step 2 — Define Search Bounds


In [ ]:
# The search operates between lower and upper channel bounds.
# It probes the midpoint between lo and hi until convergence.
from dynamic_model import DynamicNet, size_mb, param_count

lo = [8, 16, 32, 64]
hi = [64, 128, 256, 512]

print(f"Search Space Bounds:\n")
print(f"  lo: {lo} -> {size_mb(DynamicNet(lo)):.4f} MB ({param_count(DynamicNet(lo)):,} params)")
print(f"  hi: {hi} -> {size_mb(DynamicNet(hi)):.4f} MB ({param_count(DynamicNet(hi)):,} params)")


## 🎓 Step 3 — Train the Teacher (ResNet-18)
The teacher is **unconstrained in size** — it is never submitted.
Its only job is to provide soft-label knowledge to our tiny student.

> This will take ~30-60 min on T4. Run once and save to Drive.

In [ ]:
# Mount Drive to persist the teacher checkpoint
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/sos_checkpoints'
!mkdir -p {DRIVE_DIR}

In [ ]:
# ── Skip this cell if teacher.pth already exists on Drive ──────────────────
import os
TEACHER_PATH = f'{DRIVE_DIR}/teacher.pth'

if os.path.exists(TEACHER_PATH):
    print('✅ Teacher already trained. Skipping.')
else:
    !python teacher_train.py \
        --data ./data \
        --epochs 100 \
        --out {TEACHER_PATH}

## 🔍 Step 4 — Run the Architecture Binary Search

**Flags:**
| Flag | Default | Meaning |
|------|---------|----------|
| `--lo` | `8 16 32` | Lower bound configuration |
| `--hi` | `64 128 256` | Upper bound configuration |
| `--proxy-epochs` | 20 | Fast vibe-check: skip models staying below proxy-thresh |
| `--full-epochs` | 100 | Full training for models that pass the vibe check |
| `--proxy-thresh` | 0.65 | If proxy acc < 0.65, treat architecture as too small |
| `--target-acc` | 0.85 | The 85% threshold required for score points |

In [ ]:
STUDENT_OUT = f'{DRIVE_DIR}/best_student.pth'

!python search.py \
    --data ./data \
    --teacher {TEACHER_PATH} \
    --lo 8 16 32 64 \
    --hi 64 128 256 512 \
    --proxy-epochs 20 \
    --full-epochs 100 \
    --proxy-thresh 0.65 \
    --target-acc 0.85 \
    --out {STUDENT_OUT}


## 📋 Step 5 — Inspect Search Results

In [ ]:
import json

with open('search_results.json') as f:
    log = json.load(f)

print(f"{'Iter':<6} {'Config':<30} {'MB':>8} {'Proxy':>8} {'Full':>8}  {'Verdict'}")
print('─' * 85)
for r in log:
    full = f"{r['full_acc']:.4f}" if r['full_acc'] is not None else '   —  '
    print(f"{r['iteration']:<6} {str(r['config']):<30} "
          f"{r['mb']:>8.4f} {r['proxy_acc']:>8.4f} {full:>8}   {r['verdict']}")


## ✅ Step 6 — Final Size & Accuracy Check

In [ ]:
import os, torch
from torchvision import transforms
from torchvision.datasets import STL10
from torch.utils.data import DataLoader
from dynamic_model import DynamicNet, ranked_configs, size_mb
from train_recipe import VAL_TRANSFORM, evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Find the best config from log
best = [r for r in log if r['verdict'] == 'sufficient']
best.sort(key=lambda r: r['mb'])

if not best:
    print('❌ No sufficient config found. Expand the grid or increase full-epochs.')
else:
    winner = best[0]
    print(f"Winner : {winner['config']}")
    print(f"Size   : {winner['mb']:.4f} MB")
    print(f"Val Acc: {winner['full_acc']:.4f} ({winner['full_acc']*100:.2f}%)")
    print(f"{'✅ Above 85%!' if winner['full_acc'] >= 0.85 else '❌ Below 85%'}")

    # Verify on fresh loader
    val_ds = STL10(root='./data', split='test', download=True, transform=VAL_TRANSFORM)
    val_ld = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2)
    student = DynamicNet(winner['config']).to(device)
    student.load_state_dict(torch.load(STUDENT_OUT, map_location=device))
    acc = evaluate(student, val_ld, device)
    print(f'\n📊 Live verified accuracy: {acc:.4f} ({acc*100:.2f}%)')

    sz = os.path.getsize(STUDENT_OUT) / 1024**2
    print(f'📦 Actual .pth file size : {sz:.4f} MB')